In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes pillow torch torchvision tqdm sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

# Define the model ID
model_id = "llava-hf/llava-1.5-7b-hf"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

processor = LlavaProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affi

In [ ]:
import os

# Unzip MME subset
if not os.path.exists("/content/MME"):
    !unzip -q /content/MME.zip -d /content/MME

# Unzip POPE subset
if not os.path.exists("/content/POPE"):
    !unzip -q /content/POPE.zip -d /content/POPE

# Unzip MMVP subset
if not os.path.exists("/content/MMVP"):
    !unzip -q /content/MMVP.zip -d /content/MMVP

In [ ]:
import os
import json
import time
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

def generate_benchmark_answers(benchmark_dir, question_filename, output_filename, model_id_name="llava-v1.5-7b"):

    input_jsonl_path = os.path.join(benchmark_dir, "Questions", question_filename)
    output_jsonl_path = os.path.join(benchmark_dir, output_filename)
    images_dir = os.path.join(benchmark_dir, "Images")

    if not os.path.exists(input_jsonl_path):
        print(f"Warning: {input_jsonl_path} not found. Skipping...")
        return

    data = []
    with open(input_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))

    # ---------------------------------------------------------
    # DYNAMIC TOKENIZER SETUP (Same as Variants)
    # ---------------------------------------------------------
    def get_token_ids(word_list):
        raw_ids = processor.tokenizer(word_list, add_special_tokens=False).input_ids
        return set(item for sublist in raw_ids for item in (sublist if isinstance(sublist, list) else [sublist]))

    yes_ids = get_token_ids(["Yes", " Yes", "yes", " yes"])
    no_ids = get_token_ids(["No", " No", "no", " no"])
    a_ids = get_token_ids(["(a)", " (a)", "(a", " (", "(", "a", " a", "A", " A"])
    b_ids = get_token_ids(["(b)", " (b)", "(b", "b", " b", "B", " B"])

    with open(output_jsonl_path, 'w', encoding='utf-8') as outfile:
        for item in tqdm(data, desc=f"Processing {os.path.basename(benchmark_dir)}"):

            image_filename = item.get("image", "")
            prompt_text = item.get("text", item.get("question", "")).strip()

            if not prompt_text:
                continue

            img_path = os.path.join(images_dir, image_filename)
            if not os.path.exists(img_path):
                img_path = os.path.join(benchmark_dir, image_filename)

            try:
                raw_image = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"\nError loading image {img_path}: {e}. Skipping...")
                continue

            full_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"

            inputs = processor(
                text=full_prompt,
                images=raw_image,
                return_tensors="pt"
            )

            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.inference_mode():
                # ---------------------------------------------------------
                # START TIMER & GENERATE
                # ---------------------------------------------------------
                start_time = time.perf_counter()

                # output_scores=True is required to extract the logits for analysis
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    temperature=0.0,
                    do_sample=False,
                    return_dict_in_generate=True,
                    output_scores=True
                )

                end_time = time.perf_counter()
                inference_time = end_time - start_time
                # ---------------------------------------------------------

            # ---------------------------------------------------------
            # LOGIT & PROBABILITY ANALYSIS
            # ---------------------------------------------------------
            # outputs.scores[0] contains the logits for the very first generated token
            first_token_logits = outputs.scores[0][0]
            probs = F.softmax(first_token_logits.float(), dim=-1)

            is_multiple_choice = "(a)" in prompt_text.lower() and "(b)" in prompt_text.lower()

            if is_multiple_choice:
                opt1_prob = sum(probs[idx].item() for idx in a_ids if idx < probs.size(-1))
                opt2_prob = sum(probs[idx].item() for idx in b_ids if idx < probs.size(-1))
            else:
                opt1_prob = sum(probs[idx].item() for idx in yes_ids if idx < probs.size(-1))
                opt2_prob = sum(probs[idx].item() for idx in no_ids if idx < probs.size(-1))

            top_k_probs, top_k_ids = torch.topk(probs, k=5)
            top_k_tokens = processor.tokenizer.convert_ids_to_tokens(top_k_ids)
            top_k_dict = {token: round(prob.item(), 4) for token, prob in zip(top_k_tokens, top_k_probs)}
            entropy = -(probs * torch.log(probs + 1e-9)).sum().item()

            # ---------------------------------------------------------
            # DECODE OUTPUT
            # ---------------------------------------------------------
            generated_text = processor.decode(outputs.sequences[0], skip_special_tokens=True)
            response_text = generated_text.split("ASSISTANT:")[-1].strip()

            output_item = {
                "question_id": item.get("question_id", ""),
                "prompt": prompt_text,
                "label": item.get("label", ""),
                "response": response_text,
                "image": image_filename,
                "model_id": model_id_name,
                "opt1_prob": round(opt1_prob, 4),
                "opt2_prob": round(opt2_prob, 4),
                "entropy": round(entropy, 4),
                "top_5_distribution": top_k_dict,
                "inference_time": round(inference_time, 4)
            }

            outfile.write(json.dumps(output_item) + "\n")

    print(f"Finished! Answers saved to {output_jsonl_path}")

In [ ]:
benchmarks = [
    {
        "name": "MME",
        "path": "/content/MME",
        "questions_file": "mme-questions.jsonl"
    },
    {
        "name": "MMVP",
        "path": "/content/MMVP",
        "questions_file": "mmvp-questions.jsonl"
    },
    {
        "name": "POPE",
        "path": "/content/POPE",
        "questions_file": "pope-questions.jsonl"
    }
]

for bm in benchmarks:


    output_filename = f"{bm['name'].lower()}_llava_v1_5_7b_answers.jsonl"

    generate_benchmark_answers(
        benchmark_dir=bm["path"],
        question_filename=bm["questions_file"],
        output_filename=output_filename,
        model_id_name="llava-v1.5-7b"
    )


Processing MME: 100%|██████████| 200/200 [01:21<00:00,  2.45it/s]


Finished! Answers saved to /content/MME/mme_llava_v1_5_7b_answers.jsonl


Processing MMVP: 100%|██████████| 300/300 [03:50<00:00,  1.30it/s]


Finished! Answers saved to /content/MMVP/mmvp_llava_v1_5_7b_answers.jsonl


Processing POPE: 100%|██████████| 80/80 [02:28<00:00,  1.86s/it]

Finished! Answers saved to /content/POPE/pope_llava_v1_5_7b_answers.jsonl


In [ ]:
# Evaluate MME Results
import os
import json
from collections import defaultdict

result_file = "/content/MME/mme_llava_v1_5_7b_answers.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the MME jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"Evaluating MME Benchmark for: {result_file}\n")
    print("-" * 50)

    # Note: Python's json.loads natively handles the 'NaN' values generated by the entropy calculation.
    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    results_by_category = defaultdict(lambda: defaultdict(list))

    for item in answers:
        gt = str(item.get('label', '')).lower().strip()
        text = str(item.get('response', '')).lower().strip()

        # Parse Yes/No response
        if '.' in text:
            text = text.split('.')[0]
        text = text.replace(',', '').split(' ')

        if 'no' in text or 'not' in text:
            pred = 'no'
        else:
            pred = 'yes'

        is_correct = (pred == gt)

        # ---------------------------------------------------------
        # FIX: Robustly pair questions using `question_id`
        # ---------------------------------------------------------
        question_id = item.get("question_id", "")
        if "_" in question_id:
            # Splits "02598153523880aa_0" into base "02598153523880aa"
            img_id = question_id.rsplit('_', 1)[0]
        else:
            img_id = question_id

        # Determine Category
        image_path = item.get('image', 'unknown.jpg')
        if 'category' in item:
            category = item['category']
        elif '/' in image_path:
            category = image_path.split('/')[-2]
        else:
            category = 'Overall'

        results_by_category[category][img_id].append(is_correct)

    total_acc_sum = 0
    total_acc_plus_sum = 0

    print(f"{'Category':<20} | {'Accuracy':<10} | {'Accuracy+':<10} | {'Score (Max 200)':<10}")
    print("-" * 60)

    for category, images in results_by_category.items():
        total_questions = 0
        correct_questions = 0

        total_images = 0
        correct_images = 0

        for img_id, outcomes in images.items():
            total_images += 1
            total_questions += len(outcomes)
            correct_questions += sum(outcomes)

            # Accuracy+ requires getting BOTH questions for the image right
            if len(outcomes) == 2 and sum(outcomes) == 2:
                correct_images += 1

        acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
        acc_plus = (correct_images / total_images) * 100 if total_images > 0 else 0
        score = acc + acc_plus

        total_acc_sum += acc
        total_acc_plus_sum += acc_plus

        print(f"{category:<20} | {acc:<10.2f} | {acc_plus:<10.2f} | {score:<10.2f}")

    print("-" * 60)

    num_categories = len(results_by_category)
    if num_categories > 0:
        avg_acc = total_acc_sum / num_categories
        avg_acc_plus = total_acc_plus_sum / num_categories
        total_score = total_acc_sum + total_acc_plus_sum
        print(f"{'TOTAL / AVERAGE':<20} | {avg_acc:<10.2f} | {avg_acc_plus:<10.2f} | {total_score:<10.2f}")

Evaluating MME Benchmark for: /content/MME/mme_llava_v1_5_7b_answers.jsonl

--------------------------------------------------
Category             | Accuracy   | Accuracy+  | Score (Max 200)
------------------------------------------------------------
Overall              | 72.50      | 47.00      | 119.50    
------------------------------------------------------------
TOTAL / AVERAGE      | 72.50      | 47.00      | 119.50    


In [ ]:
# Evaluate POPE Results

result_file = "/content/POPE/pope_llava_v1_5_7b_answers.jsonl"

if not os.path.exists(result_file):
    print(f"Could not find the POPE jsonl file at {result_file}. Please check the path or run inference first!")
else:
    print(f"✅ Found file! Evaluating: {result_file}\n")
    print("-" * 50)

    with open(result_file, 'r', encoding='utf-8') as f:
        answers = [json.loads(line) for line in f if line.strip()]

    pred_list = []
    label_list = []

    for item in answers:

        gt = str(item.get('label', '')).lower().strip()
        label_list.append(0 if gt == 'no' else 1)


        text = str(item.get('response', '')).strip()


        if '.' in text:
            text = text.split('.')[0]

        text = text.replace(',', '')
        words = text.split(' ')


        if 'No' in words or 'not' in words or 'no' in words:
            pred_list.append(0)
        else:
            pred_list.append(1)

    pos, neg = 1, 0
    yes_ratio = pred_list.count(1) / len(pred_list) if pred_list else 0

    TP, TN, FP, FN = 0, 0, 0, 0
    for pred, label in zip(pred_list, label_list):
        if pred == pos and label == pos:
            TP += 1
        elif pred == pos and label == neg:
            FP += 1
        elif pred == neg and label == neg:
            TN += 1
        elif pred == neg and label == pos:
            FN += 1

    print('TP\tFP\tTN\tFN')
    print(f'{TP}\t{FP}\t{TN}\t{FN}\n')

    precision = float(TP) / float(TP + FP) if (TP + FP) > 0 else 0
    recall = float(TP) / float(TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0

    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1 score:  {f1:.4f}')
    print(f'Yes ratio: {yes_ratio:.4f}')
    print("-" * 50)

✅ Found file! Evaluating: /content/POPE/pope_llava_v1_5_7b_answers.jsonl

--------------------------------------------------
TP	FP	TN	FN
34	6	34	6

Accuracy:  0.8500
Precision: 0.8500
Recall:    0.8500
F1 score:  0.8500
Yes ratio: 0.5000
--------------------------------------------------


In [ ]:
import re


result_files = {
    "Model": "/content/MMVP/mmvp_llava_v1_5_7b_answers.jsonl"
}

def evaluate_mmvp_file(filepath, variant_name):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}. Skipping {variant_name}...\n")
        return

    with open(filepath, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f if line.strip()]

    total_questions = len(data)
    correct_questions = 0


    pairs_tracker = {}

    for item in data:
        question_id = int(item.get("question_id", 0))
        pair_id = (question_id + 1) // 2

        prompt = item.get("prompt", "").lower()
        label = item.get("label", "").lower().strip() # usually "(a)" or "(b)"
        response = item.get("response", "").lower().strip()


        option_a_text = ""
        option_b_text = ""

        match_a = re.search(r'\(a\)\s*(.*?)\s*\(\s*b\s*\)', prompt)
        match_b = re.search(r'\(\s*b\s*\)\s*(.*)', prompt)

        if match_a:
            option_a_text = match_a.group(1).strip()
        if match_b:
            option_b_text = match_b.group(1).strip()


        option_a_text_clean = re.sub(r'[^\w\s]', '', option_a_text)
        option_b_text_clean = re.sub(r'[^\w\s]', '', option_b_text)
        response_clean = re.sub(r'[^\w\s]', '', response)


        predicted_label = None


        if response.startswith("(a)") or response.startswith("a)") or response.startswith("a."):
            predicted_label = "(a)"
        elif response.startswith("(b)") or response.startswith("b)") or response.startswith("b."):
            predicted_label = "(b)"
        else:

            has_a = option_a_text_clean in response_clean if option_a_text_clean else False
            has_b = option_b_text_clean in response_clean if option_b_text_clean else False

            if has_a and not has_b:
                predicted_label = "(a)"
            elif has_b and not has_a:
                predicted_label = "(b)"
            else:

                opt1_prob = item.get("opt1_prob", 0.0)
                opt2_prob = item.get("opt2_prob", 0.0)
                predicted_label = "(a)" if opt1_prob > opt2_prob else "(b)"


        is_correct = (predicted_label == label)
        if is_correct:
            correct_questions += 1


        if pair_id not in pairs_tracker:
            pairs_tracker[pair_id] = []
        pairs_tracker[pair_id].append(is_correct)


    correct_pairs = 0
    total_pairs = len(pairs_tracker)

    for p_id, outcomes in pairs_tracker.items():

        if len(outcomes) == 2 and all(outcomes):
            correct_pairs += 1

    q_acc = (correct_questions / total_questions) * 100 if total_questions > 0 else 0
    p_acc = (correct_pairs / total_pairs) * 100 if total_pairs > 0 else 0


    print(f"Question-Level Accuracy: {correct_questions}/{total_questions} ({q_acc:.2f}%)")
    print(f"Pair-Level Accuracy:     {correct_pairs}/{total_pairs} ({p_acc:.2f}%)\n")



print("Evaluating MMVP Benchmarks...\n" + "="*40 + "\n")
for variant, path in result_files.items():
    evaluate_mmvp_file(path, variant)

Evaluating MMVP Benchmarks...

Question-Level Accuracy: 174/300 (58.00%)
Pair-Level Accuracy:     32/150 (21.33%)

